In [ ]:
import pandas as pd

from scouting.constants import MIN_PASSES, ROOT_PATH
from scouting.data_transform.passmaps import (
    calculate_median_positions,
    filter_on_time_window,
    get_pair_stats,
    get_passes_df,
    get_player_stats,
    get_substitutes_and_red_card_minutes,
)
from scouting.visualisations.passmaps import (
    add_details,
    plot_edges,
    plot_nodes,
    plot_pitch,
)

# Load the data into a DataFrame
events_df = pd.read_parquet(ROOT_PATH / "tests" / 'xT_salzbourg_brest.parquet')
cards_and_subs = pd.read_parquet(ROOT_PATH / "tests" / 'cards_and_subs_salzbourg_sb29_game_data.parquet')
events_df


In [2]:
first_row=events_df.iloc[0]
game_score = f"{first_row['home_team']} {first_row['home_score']} - {first_row['away_score']} {first_row['away_team']}"
game_subtitle = f"{first_row['league']} | {first_row['date'].date()}"

teams = events_df[['team_id','team']].value_counts().index.to_list()

In [ ]:
# Select team
for team_id, team_name in teams:
    team_data = events_df[events_df['team_id'] == team_id]

    minutes, subbed_off_times, subbed_on_times = get_substitutes_and_red_card_minutes(cards_and_subs, team_id, team_data)
    minutes_slices = [(0, minutes[0])] + [(minutes[i], minutes[i+1]) for i in range(len(minutes)-1)]


    for minutes_slice in minutes_slices:
        print(minutes_slice)
        passes_df = get_passes_df(team_data)
        passes_df_short = filter_on_time_window(passes_df, minutes_slice[0], minutes_slice[1], subbed_off_times, subbed_on_times)

        slice_players = passes_df_short['player'].unique()
        
        print(f"Number of players in slice: {len(slice_players)}, {slice_players}")
        # Get all successful passes and short
        passes_df_suc = passes_df[
            (passes_df['player'] != passes_df['pass_recipient_name']) &
            (passes_df['result_name'] == 'success')
        ]

        passes_df_suc_short = passes_df_short[
            (passes_df_short['player'] != passes_df_short['pass_recipient_name']) &
            (passes_df_short['result_name'] == 'success')
        ]

        # TODO : Normalize stats if we give short time period
        player_positions = calculate_median_positions(passes_df_short)


        player_stats = get_player_stats(passes_df, passes_df_suc, passes_df_suc_short, player_positions)
        pair_stats_filtered = get_pair_stats(passes_df, passes_df_suc, passes_df_suc_short, MIN_PASSES)

        player_stats = player_stats.loc[player_stats.index.map(lambda x: x in slice_players)]
        pair_stats_filtered = pair_stats_filtered[pair_stats_filtered.index.map(lambda x: (x.split('_')[0] in slice_players) & (x.split('_')[1] in slice_players))]

        fig, ax, pitch = plot_pitch()

        plot_nodes(ax, player_stats, False)
        plot_edges(ax, player_stats, pair_stats_filtered, False)


        add_details(fig, ax, game_score, game_subtitle, team_name=team_name, start_minute=minutes_slice[0], end_minute=minutes_slice[1])
        fig.savefig('./pass_map.jpeg', bbox_inches='tight', dpi=400)

## TODO

- Edit : Limit time at first substitute or red card is weird. Maybe there is something better to do like iterating on players on their whole game period